# Chapter 4 — Implementing a GPT model from scratch to generate text

This notebook assembles the components developed in earlier chapters into a GPT-style language model. The implementation begins with
a validated architecture configuration and a shape-preserving dummy model before replacing each placeholder with a real Transformer
component.

## Learning goals

- represent model hyperparameters with a typed, validated configuration;
- trace token IDs through embeddings, Transformer blocks, normalization, and logits;
- preserve `(batch, tokens, embedding)` shapes through the Transformer blocks; and
- produce one vocabulary-logit vector per input token.

## 4.1 Defining a typed and validated GPT configuration

A plain dictionary accepts misspelled keys, missing values, invalid ranges, and incompatible dimensions until much later in model
construction. `GPTConfig` replaces string-based access such as `cfg["emb_dim"]` with typed attribute access such as `cfg.emb_dim`.

Pydantic validates values when the object is created. Positive dimensions, a dropout rate in `[0, 1)`, and divisibility of `emb_dim`
by `num_heads` become explicit architecture contracts. Freezing the model prevents accidental mutation after layers are initialized.

In [1]:
from typing import Self

from pydantic import BaseModel, ConfigDict, Field, model_validator


class GPTConfig(BaseModel):
    """Validated architecture settings for a GPT-style language model."""

    # Configuration objects behave like immutable architecture specifications.
    model_config = ConfigDict(frozen=True)

    vocab_size: int = Field(gt=0, description="Number of tokenizer vocabulary entries")
    context_length: int = Field(gt=0, description="Maximum supported token sequence length")
    emb_dim: int = Field(gt=0, description="Token embedding and model dimension, also called d_model")
    num_heads: int = Field(gt=0, description="Number of parallel attention heads")
    num_layers: int = Field(gt=0, description="Number of Transformer blocks")
    dropout_rate: float = Field(ge=0.0, lt=1.0, description="Dropout probability")
    qkv_bias: bool = Field(description="Whether QKV projections include bias terms")

    @model_validator(mode="after")
    def validate_attention_dimensions(self) -> Self:
        """Require every attention head to receive an equal feature width."""
        if self.emb_dim % self.num_heads != 0:
            raise ValueError("emb_dim must be divisible by num_heads")
        return self

    @property
    def head_dim(self) -> int:
        """Return the query, key, and value width assigned to one head."""
        return self.emb_dim // self.num_heads


# GPT-2 small / 124M uses a model dimension of 768 and 12 attention heads.
GPT_CONFIG_124M = GPTConfig(
    vocab_size=50257,
    context_length=1024,
    emb_dim=768,
    num_heads=12,
    num_layers=12,
    dropout_rate=0.1,
    qkv_bias=False,
)

### GPT-2 small configuration

`GPT_CONFIG_124M` describes the smallest GPT-2 architecture: a 50,257-token vocabulary, context length 1,024, embedding dimension 768,
12 heads, and 12 Transformer blocks. Its derived `head_dim` is `768 / 12 = 64`.

Use `GPT_CONFIG_124M.model_dump()` only when an external API specifically requires a dictionary; model code should prefer typed
attributes.

## 4.2 Creating shape-preserving placeholders

Before implementing full Transformer blocks and layer normalization, identity modules let us assemble and test the outer GPT data
flow. They expose the same typed interfaces as the eventual components while returning inputs unchanged.

In [2]:
import torch
from torch import nn


class DummyTransformerBlock(nn.Module):
    """Placeholder Transformer block that preserves its input unchanged."""

    def __init__(self, cfg: GPTConfig) -> None:
        """Accept the final block interface without creating operations yet."""
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return a tensor with the same values and shape as the input."""
        return x


class DummyLayerNorm(nn.Module):
    """Placeholder final normalization that acts as an identity function."""

    def __init__(self, normalized_shape: int, eps: float = 1e-5) -> None:
        """Accept the final normalization interface without parameters yet."""
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return a tensor with the same values and shape as the input."""
        return x

C:\Users\giloz\dev\build-llms-from-scratch-companion\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## 4.3 Assembling the dummy GPT model

The model maps token IDs through these stages:

```text
token IDs                     (B, T)
token embeddings              (B, T, emb_dim)
positional embeddings         (T, emb_dim)
combined token representations (B, T, emb_dim)
Transformer block stack       (B, T, emb_dim)
final normalization           (B, T, emb_dim)
vocabulary logits             (B, T, vocab_size)
```

Positional embeddings broadcast across the batch. The output head changes only the final feature dimension, producing one vector of
next-token scores for every input position. During training, all positions are predicted in parallel, which makes the
computation efficient; autoregressive generation typically uses only the final position.

In [3]:
class DummyGPTModel(nn.Module):
    """Trace GPT tensor shapes while Transformer internals remain placeholders."""

    def __init__(self, cfg: GPTConfig) -> None:
        """Create embeddings, placeholder blocks, normalization, and output head."""
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.emb_dim)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.emb_dim)
        self.drop_emb = nn.Dropout(cfg.dropout_rate)
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg.num_layers)]
        )
        self.final_norm = DummyLayerNorm(cfg.emb_dim)
        self.out_head = nn.Linear(cfg.emb_dim, cfg.vocab_size, bias=False)

    def forward(self, in_idx: torch.Tensor) -> torch.Tensor:
        """Map token IDs `(B, T)` to vocabulary logits `(B, T, vocab_size)`."""
        # in_idx: (batch_size, num_tokens)
        _, num_tokens = in_idx.shape

        # tok_embeds: (batch_size, num_tokens, emb_dim)
        tok_embeds = self.tok_emb(in_idx)
        # pos_embeds: (num_tokens, emb_dim), broadcast across the batch dimension
        pos_embeds = self.pos_emb(
            torch.arange(num_tokens, device=in_idx.device)
        )

        # x: (batch_size, num_tokens, emb_dim)
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        # Placeholder blocks and normalization preserve x's shape.
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        # One next-token prediction per position, computed in parallel.
        # logits: (batch_size, num_tokens, vocab_size)
        logits = self.out_head(x)
        return logits

## Checkpoint

The architecture is now represented by an immutable, validated `GPTConfig`, and the dummy model establishes the complete tensor-shape
contract from token IDs to vocabulary logits. Subsequent sections can replace placeholders without changing that outer interface.

In [4]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
batch

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [5]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)
